# Graph Construction for LSTM-GCN Options Trading

This notebook implements **Section 5: Proposed Methodology** from the paper, specifically the graph construction methods:

1. **Pearson Correlation Method (Section 5.2.1)** - Build adjacency matrix from correlation of underlying equity returns
2. **Convex Optimization Method (Section 5.2.2)** - Learn adjacency matrix via optimization using PyTorch

## Outputs
- `processed_data/A_pearson.npy` - Pearson correlation-based adjacency matrix
- `processed_data/A_convex.npy` - Convex optimization-based adjacency matrix  
- `processed_data/graph_metadata.json` - Parameters and metrics

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, 'src')

from graph_construction import (
    # Pearson method
    compute_log_returns,
    compute_correlation_matrix,
    build_pearson_adjacency,
    sweep_pearson_thresholds,
    # Convex optimization
    learn_adjacency_convex,
    grid_search_convex,
    build_graph_ensemble,
    normalize_adjacency,
    # Metrics
    compute_connectivity,
    compute_edge_homophily,
    compute_louvain_modularity,
    compute_graph_metrics,
    # Utilities
    reshape_feature_tensor_for_graph,
    create_sector_labels,
)

print("Imports successful!")

In [ ]:
# Configuration
DATA_DIR = 'processed_data'
OUTPUT_DIR = 'processed_data'

# Check for PyTorch
try:
    import torch
    TORCH_AVAILABLE = True
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not available, will use NumPy fallback")

In [ ]:
# Load processed data from data_processing.ipynb
print("Loading processed data...")

# Load metadata
with open(f'{DATA_DIR}/metadata.json', 'r') as f:
    metadata = json.load(f)

dims = metadata['dimensions']
print(f"Dataset: {dims['N']} assets, {dims['delta']} days")
print(f"Sectors: {list(metadata['sector_map'].values())[:5]}...")

In [ ]:
# Load equity prices for Pearson correlation
try:
    equity_df = pd.read_csv(f'{DATA_DIR}/equity_prices.csv', index_col=0, parse_dates=True)
    print(f"Equity prices loaded: {equity_df.shape}")
except FileNotFoundError:
    print("ERROR: equity_prices.csv not found!")
    print("Please re-run data_processing.ipynb to generate this file.")
    raise

# Load feature tensor for convex optimization
feature_tensor = np.load(f'{DATA_DIR}/feature_tensor.npy')
print(f"Feature tensor loaded: {feature_tensor.shape}")

# Load feature names
feature_names = metadata['feature_names']
print(f"Features: {feature_names}")

In [ ]:
# Extract ticker list and sector labels
tickers = metadata['tickers']
sector_map = metadata['sector_map']

# Create numeric sector labels for homophily computation
sector_labels, sector_to_id = create_sector_labels(tickers, sector_map)

print(f"\nTickers ({len(tickers)}): {tickers[:5]}...")
print(f"\nSector mapping:")
for sector, idx in sector_to_id.items():
    count = np.sum(sector_labels == idx)
    print(f"  {sector}: {count} assets")

## 2. Pearson Correlation Method (Section 5.2.1)

Build adjacency matrix from correlation of underlying equity log-returns.

**Equation (40):**
$$\rho_{ij} = \frac{\sum_t (r_i^t - \bar{r}_i)(r_j^t - \bar{r}_j)}{\sqrt{\sum_t (r_i^t - \bar{r}_i)^2 \cdot \sum_t (r_j^t - \bar{r}_j)^2}}$$

**Equation (41):**
$$A_{ij} = \begin{cases} \tau & \text{if } |\rho_{ij}| \geq \tau \text{ and } i \neq j \\ 0 & \text{otherwise} \end{cases}$$

In [ ]:
# Compute log-returns from equity prices
log_returns = compute_log_returns(equity_df)
print(f"Log-returns shape: {log_returns.shape}")
print(f"\nLog-returns statistics:")
print(log_returns.describe().T[['mean', 'std', 'min', 'max']].head(10))

In [ ]:
# Compute Pearson correlation matrix
corr_matrix = compute_correlation_matrix(log_returns)
print(f"Correlation matrix shape: {corr_matrix.shape}")

# Plot correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Correlation')
ax.set_title('Pearson Correlation Matrix (Equity Log-Returns)')
ax.set_xlabel('Asset Index')
ax.set_ylabel('Asset Index')
plt.tight_layout()
plt.show()

# Statistics
upper_tri = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
print(f"\nCorrelation statistics (upper triangle):")
print(f"  Mean: {np.mean(upper_tri):.4f}")
print(f"  Std: {np.std(upper_tri):.4f}")
print(f"  Min: {np.min(upper_tri):.4f}")
print(f"  Max: {np.max(upper_tri):.4f}")

In [ ]:
# Sweep thresholds to find optimal tau
thresholds = np.arange(0.30, 0.75, 0.05)
threshold_results = sweep_pearson_thresholds(corr_matrix, thresholds, sector_labels)

print("Threshold sweep results:")
print(threshold_results.to_string(index=False))

In [ ]:
# Plot metrics vs threshold (Figure 7 from paper)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Connectivity
ax = axes[0, 0]
ax.plot(threshold_results['threshold'], threshold_results['connectivity'], 'b-o')
ax.set_xlabel('Threshold (τ)')
ax.set_ylabel('Algebraic Connectivity (λ₂)')
ax.set_title('Graph Connectivity vs Threshold')
ax.grid(True, alpha=0.3)

# Number of edges
ax = axes[0, 1]
ax.plot(threshold_results['threshold'], threshold_results['num_edges'], 'g-o')
ax.set_xlabel('Threshold (τ)')
ax.set_ylabel('Number of Edges')
ax.set_title('Graph Sparsity vs Threshold')
ax.grid(True, alpha=0.3)

# Edge homophily
ax = axes[1, 0]
ax.plot(threshold_results['threshold'], threshold_results['homophily'], 'r-o')
ax.set_xlabel('Threshold (τ)')
ax.set_ylabel('Edge Homophily')
ax.set_title('Sector Homophily vs Threshold')
ax.grid(True, alpha=0.3)

# Louvain modularity
ax = axes[1, 1]
ax.plot(threshold_results['threshold'], threshold_results['modularity'], 'm-o')
ax.set_xlabel('Threshold (τ)')
ax.set_ylabel('Louvain Modularity')
ax.set_title('Community Structure vs Threshold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/pearson_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Select optimal threshold (maximize modularity)
best_idx = threshold_results['modularity'].idxmax()
optimal_tau = threshold_results.loc[best_idx, 'threshold']
print(f"\nOptimal threshold (max modularity): τ = {optimal_tau:.2f}")
print(f"  Modularity: {threshold_results.loc[best_idx, 'modularity']:.4f}")
print(f"  Connectivity: {threshold_results.loc[best_idx, 'connectivity']:.4f}")
print(f"  Homophily: {threshold_results.loc[best_idx, 'homophily']:.4f}")
print(f"  Num edges: {threshold_results.loc[best_idx, 'num_edges']}")

In [ ]:
# Build final Pearson adjacency matrix
A_pearson = build_pearson_adjacency(corr_matrix, optimal_tau)

# Compute comprehensive metrics
pearson_metrics = compute_graph_metrics(A_pearson, sector_labels)

print("\n=== Pearson Correlation Adjacency Matrix ===")
print(f"Shape: {A_pearson.shape}")
for key, val in pearson_metrics.items():
    if isinstance(val, float):
        print(f"  {key}: {val:.4f}")
    else:
        print(f"  {key}: {val}")

In [ ]:
# Visualize Pearson adjacency
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Adjacency matrix
ax = axes[0]
im = ax.imshow(A_pearson, cmap='Blues')
plt.colorbar(im, ax=ax, label='Edge Weight')
ax.set_title(f'Pearson Adjacency Matrix (τ={optimal_tau:.2f})')
ax.set_xlabel('Asset Index')
ax.set_ylabel('Asset Index')

# Degree distribution
ax = axes[1]
degrees = A_pearson.sum(axis=1)
ax.hist(degrees, bins=20, edgecolor='black', alpha=0.7)
ax.axvline(degrees.mean(), color='r', linestyle='--', label=f'Mean: {degrees.mean():.2f}')
ax.set_xlabel('Node Degree')
ax.set_ylabel('Frequency')
ax.set_title('Degree Distribution (Pearson)')
ax.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/A_pearson_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Convex Optimization Method (Section 5.2.2)

Learn adjacency matrix by minimizing:

**Equation (42):**
$$\min_A \text{tr}(X^T(D-A)X) - \alpha \cdot \mathbf{1}^T \log(A\mathbf{1}) + \beta \|A\|_F^2$$
$$\text{s.t. } A = A^T, \quad A_{ij} \geq 0 \; \forall i \neq j$$

Where:
- First term: Laplacian smoothness (similar nodes connected)
- Second term: Connectivity regularizer  
- Third term: Sparsity regularizer

In [ ]:
# Reshape feature tensor for graph learning
# From (delta, F, N) to (N, F*delta)
X = reshape_feature_tensor_for_graph(feature_tensor)

# Handle NaN values
X = np.nan_to_num(X, nan=0.0)

print(f"Feature matrix X shape: {X.shape}")
print(f"  N (assets): {X.shape[0]}")
print(f"  F*delta (features): {X.shape[1]}")

In [ ]:
# Grid search over alpha and beta
print("Running grid search over (alpha, beta) parameters...")
print("This may take a few minutes...\n")

alphas = [0.1, 1.0, 10.0, 100.0]
betas = [0.01, 0.1, 1.0, 10.0]

convex_results = grid_search_convex(
    X, 
    alphas=alphas, 
    betas=betas, 
    sector_labels=sector_labels,
    max_iter=5000,
    verbose=True
)

print("\nGrid search results:")
print(convex_results.to_string(index=False))

In [ ]:
# Plot grid search results (Figure 8 from paper)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Reshape for heatmap
metrics_to_plot = ['connectivity', 'homophily', 'modularity', 'num_edges']
titles = ['Algebraic Connectivity', 'Edge Homophily', 'Louvain Modularity', 'Number of Edges']

for ax, metric, title in zip(axes.flat, metrics_to_plot, titles):
    # Pivot to matrix form
    pivot = convex_results.pivot(index='alpha', columns='beta', values=metric)
    
    im = ax.imshow(pivot.values, cmap='viridis', aspect='auto')
    plt.colorbar(im, ax=ax)
    
    ax.set_xticks(range(len(betas)))
    ax.set_xticklabels([f'{b}' for b in betas])
    ax.set_yticks(range(len(alphas)))
    ax.set_yticklabels([f'{a}' for a in alphas])
    ax.set_xlabel('β (sparsity)')
    ax.set_ylabel('α (connectivity)')
    ax.set_title(title)
    
    # Add values as text
    for i in range(len(alphas)):
        for j in range(len(betas)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                text_color = 'white' if val < pivot.values.mean() else 'black'
                ax.text(j, i, f'{val:.2f}', ha='center', va='center', 
                       color=text_color, fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/convex_grid_search.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Select optimal (alpha, beta) - maximize modularity
best_idx = convex_results['modularity'].idxmax()
optimal_alpha = convex_results.loc[best_idx, 'alpha']
optimal_beta = convex_results.loc[best_idx, 'beta']

print(f"\nOptimal parameters (max modularity):")
print(f"  α = {optimal_alpha}")
print(f"  β = {optimal_beta}")
print(f"\nMetrics at optimal:")
print(f"  Modularity: {convex_results.loc[best_idx, 'modularity']:.4f}")
print(f"  Connectivity: {convex_results.loc[best_idx, 'connectivity']:.4f}")
print(f"  Homophily: {convex_results.loc[best_idx, 'homophily']:.4f}")
print(f"  Num edges: {convex_results.loc[best_idx, 'num_edges']}")

In [ ]:
# Build ensemble adjacency matrix (Equation 43)
print("\nBuilding ensemble graph from multiple time periods...")

# Define ensemble periods (1-5 years in trading days)
delta = feature_tensor.shape[0]
periods = [252, 504, 756, 1008, 1260]
periods = [p for p in periods if p <= delta]  # Only use periods we have data for

print(f"Using {len(periods)} periods: {periods}")

A_convex = build_graph_ensemble(
    feature_tensor,
    alpha=optimal_alpha,
    beta=optimal_beta,
    K=len(periods),
    periods=periods,
    max_iter=5000,
    verbose=True
)

print(f"\nEnsemble adjacency matrix shape: {A_convex.shape}")

In [ ]:
# Compute metrics for convex optimization graph
convex_metrics = compute_graph_metrics(A_convex, sector_labels)

print("\n=== Convex Optimization Adjacency Matrix ===")
print(f"Shape: {A_convex.shape}")
for key, val in convex_metrics.items():
    if isinstance(val, float):
        print(f"  {key}: {val:.4f}")
    else:
        print(f"  {key}: {val}")

In [ ]:
# Visualize convex adjacency
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Adjacency matrix
ax = axes[0]
im = ax.imshow(A_convex, cmap='Blues')
plt.colorbar(im, ax=ax, label='Edge Weight')
ax.set_title(f'Convex Optimization Adjacency (α={optimal_alpha}, β={optimal_beta})')
ax.set_xlabel('Asset Index')
ax.set_ylabel('Asset Index')

# Degree distribution
ax = axes[1]
degrees = A_convex.sum(axis=1)
ax.hist(degrees, bins=20, edgecolor='black', alpha=0.7)
ax.axvline(degrees.mean(), color='r', linestyle='--', label=f'Mean: {degrees.mean():.2f}')
ax.set_xlabel('Node Degree')
ax.set_ylabel('Frequency')
ax.set_title('Degree Distribution (Convex)')
ax.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/A_convex_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Comparison and Evaluation

In [ ]:
# Side-by-side comparison
comparison = pd.DataFrame({
    'Metric': list(pearson_metrics.keys()),
    'Pearson': list(pearson_metrics.values()),
    'Convex': list(convex_metrics.values())
})

print("=" * 60)
print("COMPARISON: Pearson vs Convex Optimization")
print("=" * 60)
print(comparison.to_string(index=False))

In [ ]:
# Visual comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Pearson adjacency
ax = axes[0, 0]
im = ax.imshow(A_pearson, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_title(f'Pearson (τ={optimal_tau:.2f})')

# Convex adjacency
ax = axes[0, 1]
im = ax.imshow(A_convex, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_title(f'Convex (α={optimal_alpha}, β={optimal_beta})')

# Degree comparison
ax = axes[1, 0]
degrees_p = A_pearson.sum(axis=1)
degrees_c = A_convex.sum(axis=1)
ax.scatter(degrees_p, degrees_c, alpha=0.6)
ax.plot([0, max(degrees_p.max(), degrees_c.max())], 
        [0, max(degrees_p.max(), degrees_c.max())], 'r--', label='y=x')
ax.set_xlabel('Pearson Degree')
ax.set_ylabel('Convex Degree')
ax.set_title('Node Degree Comparison')
ax.legend()

# Edge weight distributions
ax = axes[1, 1]
weights_p = A_pearson[A_pearson > 0].flatten()
weights_c = A_convex[A_convex > 0].flatten()
ax.hist(weights_p, bins=30, alpha=0.5, label='Pearson', density=True)
ax.hist(weights_c, bins=30, alpha=0.5, label='Convex', density=True)
ax.set_xlabel('Edge Weight')
ax.set_ylabel('Density')
ax.set_title('Edge Weight Distributions')
ax.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/graph_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Community detection comparison
_, pearson_communities = compute_louvain_modularity(A_pearson)
_, convex_communities = compute_louvain_modularity(A_convex)

print("Community Detection Results:")
print(f"\nPearson graph:")
print(f"  Number of communities: {len(set(pearson_communities))}")
print(f"  Community sizes: {np.bincount(pearson_communities)}")

print(f"\nConvex graph:")
print(f"  Number of communities: {len(set(convex_communities))}")
print(f"  Community sizes: {np.bincount(convex_communities)}")

print(f"\nGround truth (sectors):")
print(f"  Number of sectors: {len(set(sector_labels))}")
print(f"  Sector sizes: {np.bincount(sector_labels)}")

## 5. Save Outputs

In [ ]:
# Normalize adjacency matrices for GCN input
A_pearson_norm = normalize_adjacency(A_pearson, add_self_loops=True)
A_convex_norm = normalize_adjacency(A_convex, add_self_loops=True)

print("Normalized adjacency matrices for GCN:")
print(f"  A_pearson_norm: sum={A_pearson_norm.sum():.2f}")
print(f"  A_convex_norm: sum={A_convex_norm.sum():.2f}")

In [ ]:
# Save adjacency matrices
np.save(f'{OUTPUT_DIR}/A_pearson.npy', A_pearson)
np.save(f'{OUTPUT_DIR}/A_convex.npy', A_convex)
np.save(f'{OUTPUT_DIR}/A_pearson_normalized.npy', A_pearson_norm)
np.save(f'{OUTPUT_DIR}/A_convex_normalized.npy', A_convex_norm)

print("Saved:")
print(f"  {OUTPUT_DIR}/A_pearson.npy")
print(f"  {OUTPUT_DIR}/A_convex.npy")
print(f"  {OUTPUT_DIR}/A_pearson_normalized.npy")
print(f"  {OUTPUT_DIR}/A_convex_normalized.npy")

In [ ]:
# Save graph metadata
graph_metadata = {
    'pearson': {
        'method': 'Pearson correlation of equity log-returns',
        'threshold': float(optimal_tau),
        'metrics': {k: float(v) if isinstance(v, (int, float, np.floating)) else v 
                   for k, v in pearson_metrics.items()}
    },
    'convex': {
        'method': 'Convex optimization with PyTorch',
        'alpha': float(optimal_alpha),
        'beta': float(optimal_beta),
        'ensemble_periods': periods,
        'max_iter': 5000,
        'metrics': {k: float(v) if isinstance(v, (int, float, np.floating)) else v 
                   for k, v in convex_metrics.items()}
    },
    'sector_to_id': sector_to_id,
    'num_assets': int(A_pearson.shape[0]),
}

with open(f'{OUTPUT_DIR}/graph_metadata.json', 'w') as f:
    json.dump(graph_metadata, f, indent=2)

print(f"\nSaved: {OUTPUT_DIR}/graph_metadata.json")

In [ ]:
# Summary
print("\n" + "=" * 60)
print("GRAPH CONSTRUCTION COMPLETE")
print("=" * 60)
print(f"\nOutputs saved to {OUTPUT_DIR}/")
print("\nAdjacency matrices:")
print(f"  - A_pearson.npy: Correlation-based ({pearson_metrics['num_edges']} edges)")
print(f"  - A_convex.npy: Optimization-based ({convex_metrics['num_edges']} edges)")
print(f"  - Normalized versions (*_normalized.npy) for GCN input")
print("\nVisualizations:")
print(f"  - pearson_threshold_sweep.png")
print(f"  - convex_grid_search.png")
print(f"  - A_pearson_visualization.png")
print(f"  - A_convex_visualization.png")
print(f"  - graph_comparison.png")